# Assignment 1: Deep Reinforcement Learning

## Overview

In this assignment, we tackle a sequential decision-making problem using reinforcement learning (RL), without prior knowledge of the environment dynamics. Concretely, we learn a policy for an instance of the Elevator environment, which models evening rush hours when people from different floors want to go down to the ground floor. See `Elevator.ipynb` for details.

We will employ deep reinforcement learning via an **actor–critic** method with **Proximal Policy Optimization (PPO)** [1]. The actor represents a (stochastic) policy, while the critic learns a value function to provide a low-variance learning signal. PPO stabilizes policy-gradient updates by optimizing a clipped surrogate objective, yielding performance similar to TRPO while being simpler to implement.

Throughout this assignment, you will build an Actor–Critic agent with neural networks for both the policy and the value function, derive a categorical (discrete) policy from actor logits, and implement key PPO components such as advantage estimation and the PPO loss.

[1] Schulman, J., Wolski, F., Dhariwal, P., Radford, A., & Klimov, O. (2017). Proximal Policy Optimization Algorithms. arXiv preprint arXiv:1707.06347. <https://arxiv.org/abs/1707.06347>.

## Submission

By the end of this assignment, you’ll have developed an agent capable of performing reasonably in the Elevator environment. You must submit your agent to [Coursemology](https://coursemology.org/courses/3233/assessments/87603).

<span style="color: red">**Note:** Only one submission is permitted per group. If multiple submissions are received from the same group, the last one will be considered. Additionally, any late submissions will result in a penalty for the entire group.</span>

### Grading

This assignment is autograded.

**Marks:**

- Task 1: 5 marks, 1 mark for each sub-task
- Task 2: 1 mark
- Task 3: 4 marks, 1 mark for each sub-task
- Task 4: 5 marks

### Notes

- Only press “Finalise Submission” once you’ve completed all the tasks.
- The assignment won’t be auto-submitted on the deadline.

## Setup

### Google Colab

If you are using Google Colab (and we encourage you to do so), please run the following code cell. If you are not using Google Colab, you can skip this code cell.

**Note**: If your assignment folder in Google Drive is located at `My Drive -> CSxx46-Assignment-1`, you should set `CODE_DIR` to `'MyDrive/CSxx46-Assignment-1'`.

In [ ]:
import os
import sys

DRIVE_DIR = '/content/drive/' # Google Colab mount point, do not change
CODE_DIR = 'MyDrive/CSxx46-Assignment-1' # Folder in your Google Drive where the code is stored

try:
    from google.colab import drive
    drive.mount(DRIVE_DIR)
    sys.path.append(os.path.join(DRIVE_DIR, CODE_DIR))
    # Update based on the Google Drive directory
    %cd /content/drive/MyDrive/CSxx46-Assignment-1
except:
    print("Not in Colab environment, skipping drive mount")
    pass

#### Installing Dependencies

The elevator task is implemented using the `PyRDDLGym` library. Before we begin, please install the following packages.

**Note**: You may need to restart the session. Please follow the prompt to do so.

In [ ]:
!pip install pyRDDLGym==2.6
!pip install rddlrepository==2.1

### Anaconda

If you prefer Anaconda/Miniconda, you can create the exact course environment from the provided `environment.yml`.

1. Install Anaconda (or Miniconda) and open an **Anaconda Prompt** (Windows) / terminal (macOS/Linux).
2. From the assignment folder, create the environment:
   ```bash
   conda env create -f environment.yml
   ```
3. Activate it:
   ```bash
   conda activate csxx46-ay2526s2-assignment-1
   ```
4. Launch JupyterLab (or VS Code) inside the env:
   ```bash
   jupyter lab
   ```

Useful links:
- Anaconda installation guide: https://docs.anaconda.com/anaconda/install/
- Anaconda Navigator getting started: https://docs.anaconda.com/navigator/

## Import Dependencies

In [ ]:
import pyRDDLGym
from pyRDDLGym.core.env import RDDLEnv

from utils import DictToListWrapper, live_plot

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
import tqdm

from torch.distributions.categorical import Categorical

import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
from gymnasium.vector import SyncVectorEnv

## Environment

To ease environment creation in the subsequent steps, we define a function that returns an instance of the environment below. We will also include the `DictToListWrapper`, as we did in the `ElevatorEnv` notebook. Additionally, we will incorporate the `RecordEpisodeStatistics` wrapper to help track episode statistics.

In [ ]:
def create_elevator_env():
    env = RDDLEnv(
        domain="elevator/domain.rddl",
        instance="elevator/instance.rddl",
    )
    # If your observation is a Dict of booleans, flatten it:
    env = DictToListWrapper(env)
    env = RecordEpisodeStatistics(env)
    return env

### Vectorized Environments

To improve training efficiency and reduce the correlation between samples in a single sequence of experiences, we utilize multiple parallel environments to collect data. At each step, the agent interacts with $N$ environments simultaneously, storing the collected transitions in a rollout buffer. These transitions are then used to update the policy and value function.

We will use the `SyncVectorEnv` class to create multiple environments. Here, we set the number of parallel environments to 4.

**DO NOT MODIFY THE CODE BELOW**

In [ ]:
NUM_ENVS = 4

envs = SyncVectorEnv([lambda: create_elevator_env() for _ in range(NUM_ENVS)])

Our vectorized environments have identical observation and action spaces, as shown below.

In [ ]:
env = envs.envs[0].env
print(f"Observation space: {env.observation_space}")
env.get_state_description()

print(f"Action space: {env.action_space}")
env.get_action_description()

## Hyperparameters

Here, we define the hyperparameters for the algorithm. The random seed is fixed to ensure reproducibility.

**DO NOT MODIFY THE CODE BELOW**

In [ ]:
LEARNING_RATE = 2.5e-4

ROLLOUT_STEPS = 128
NUM_MINI_BATCHES = NUM_EPOCHS = 4
TOTAL_STEPS = 800000

GAMMA = 0.99
GAE_LAMBDA = 0.95

CLIP_COEF = 0.2
VALUE_LOSS_COEF = 0.5
ENTROPY_COEF = 0.01

# RANDOM SEED, DON'T MODIFY
SEED = 2048
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Helper Functions

In [ ]:
def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    """Initialize the weights and biases of a layer.

    Args:
        layer (nn.Module): The layer to initialize.
        std (float): Standard deviation for orthogonal initialization.
        bias_const (float): Constant value for bias initialization.

    Returns:"
        nn.Module: The initialized layer.
    """
    torch.nn.init.orthogonal_(layer.weight, std)  # Orthogonal initialization
    torch.nn.init.constant_(layer.bias, bias_const)  # Constant bias
    return layer


## Notation 

We use $S_t$ for the state at time $t$, $A_t$ for the action at time $t$, and $R_{t+1}$ for the reward at time $t+1$ (the reward observed after taking $A_t$ in $S_t$). The return following time $t$ is $G_t$. A stochastic policy is written as $\pi(a|s,\boldsymbol{\theta})$, and an approximate state-value function is written as $\hat v(s,\mathbf{w})$. An estimate of expectation is written as $\hat{\mathbb{E}}$.

## Actor-Critic Agent

An **actor–critic** agent combines two learned objects:

- **Actor**: a parameterized stochastic policy $\pi(a|s,\boldsymbol{\theta})$ that selects actions (here, a categorical distribution over discrete actions).
- **Critic**: a value function estimator. The true state-value function for a policy $\pi$ is
  $$
  v_\pi(s) \doteq \mathbb{E}_\pi\left[G_t \mid S_t=s\right].
  $$
  We represent the critic with a function approximator $\hat v(s,\mathbf{w})$ intended to approximate $v_\pi(s)$.

The critic provides a bootstrap-based learning signal (a temporal-difference error) that typically has much lower variance than using full Monte Carlo returns, which helps the actor improve more efficiently.

### Task 1: Implement the Actor-Critic Agent

In this task, you will complete a partially defined actor–critic agent template.

#### Task 1.1: Actor and Critic Networks

The agent consists of two neural networks:

- **Actor Network**: produces logits that parameterize $\pi(\cdot|s,\boldsymbol{\theta})$.
- **Critic Network**: outputs a scalar $\hat v(s,\mathbf{w})$, an estimate of the state value.

You will need to specify the input and output dimensions for both networks based on the environment:

- For the actor, define `actor_input_dim` and `actor_output_dim`.
- For the critic, define `critic_input_dim` and `critic_output_dim`.

**WARNING:** Do not use any variables defined outside of `__init__`. Please hardcode the dimensions.

#### Task 1.2: Value

Implement `get_value` to return a batch of value predictions $\hat v(s,\mathbf{w})$.

#### Task 1.3: Action Distribution

Implement `get_probs` to return a categorical distribution over actions for a batch of states. (Even though the function is named `get_probs`, you may construct the distribution from logits, which is numerically stable.)

**Hint**: Use [`Categorical`](https://pytorch.org/docs/stable/distributions.html#categorical).

#### Task 1.4: Sample Action

Implement `get_action` to sample $A_t \sim \pi(\cdot|S_t,\boldsymbol{\theta})$.

#### Task 1.5: Log Probability

Implement `get_action_logprob` to compute $\log \pi(A_t|S_t,\boldsymbol{\theta})$ for given actions.

In [ ]:
class ACAgent(nn.Module):
    """Actor-Critic agent using neural networks for policy and value function approximation."""

    def __init__(self):
        """Initialize the Actor-Critic agent with actor and critic networks."""
        super().__init__()

        ### ------------- TASK 1.1 ----------- ###
        """
        Define:
        - actor_input_dim: Input dimension for the actor network.
        - actor_output_dim: Output dimension for the actor network (number of actions).
        - critic_input_dim: Input dimension for the critic network.
        - critic_output_dim: Output dimension for the critic network (value estimate).
        """
        # YOUR CODE STARTS HERE
        raise NotImplementedError()
        # YOUR CODE ENDS HERE

        # Define the actor network
        self.actor = nn.Sequential(
            layer_init(nn.Linear(actor_input_dim, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 128)),
            nn.Tanh(),
            layer_init(nn.Linear(128, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, actor_output_dim), std=0.01),  # Final layer with small std for output
        )

        # Define the critic network
        self.critic = nn.Sequential(
            layer_init(nn.Linear(critic_input_dim, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 128)),
            nn.Tanh(),
            layer_init(nn.Linear(128, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, critic_output_dim), std=1.0),  # Standard output layer for value
        )

    def get_value(self, x):
        """Calculate the estimated value for a given state.

        Args:
            x (torch.Tensor): Input state, shape: (batch_size, observation_size)

        Returns:
            torch.Tensor: Estimated value for the state, shape: (batch_size, 1)
        """
        ### ------------- TASK 1.2 ----------- ###
        # YOUR CODE STARTS HERE
        raise NotImplementedError()
        # YOUR CODE ENDS HERE

    def get_probs(self, x):
        """Calculate the action probabilities for a given state.

        Args:
            x (torch.Tensor): Input state, shape: (batch_size, observation_size)

        Returns:
            torch.distributions.Categorical: Categorical distribution over actions.
        """
        ### ------------- TASK 1.3 ----------- ###
        # YOUR CODE STARTS HERE
        raise NotImplementedError()
        # YOUR CODE ENDS HERE

    def get_action(self, probs):
        """Sample an action from the action probabilities.

        Args:
            probs (torch.distributions.Categorical): Action probabilities.

        Returns:
            torch.Tensor: Sampled action, shape: (batch_size, 1)
        """
        ### ------------- TASK 1.4 ----------- ###
        # YOUR CODE STARTS HERE
        raise NotImplementedError()
        # YOUR CODE ENDS HERE

    def get_action_logprob(self, probs, action):
        """Compute the log probability of a given action.

        Args:
            probs (torch.distributions.Categorical): Action probabilities.
            action (torch.Tensor): Selected action, shape: (batch_size, 1)

        Returns:
            torch.Tensor: Log probability of the action, shape: (batch_size, 1)
        """
        ### ------------- TASK 1.5 ----------- ###
        # YOUR CODE STARTS HERE
        raise NotImplementedError()
        # YOUR CODE ENDS HERE

    def get_entropy(self, probs):
        """Calculate the entropy of the action distribution.

        Args:
            probs (torch.distributions.Categorical): Action probabilities.

        Returns:
            torch.Tensor: Entropy of the distribution, shape: (batch_size, 1)
        """
        return probs.entropy()  # Return the entropy of the probabilities

    def get_action_logprob_entropy(self, x):
        """Get action, log probability, and entropy for a given state.

        Args:
            x (torch.Tensor): Input state.

        Returns:
            tuple: (action, logprob, entropy)
                - action (torch.Tensor): Sampled action.
                - logprob (torch.Tensor): Log probability of the action.
                - entropy (torch.Tensor): Entropy of the action distribution.
        """
        probs = self.get_probs(x)  # Get the action probabilities
        action = self.get_action(probs)  # Sample an action
        logprob = self.get_action_logprob(probs, action)  # Compute log probability of the action
        entropy = self.get_entropy(probs)  # Compute entropy of the action distribution
        return action, logprob, entropy  # Return action, log probability, and entropy

    def act(self, state):
        """Select an action for a given state.

        Args:
            state (torch.Tensor): Input state.
        
        Returns:
            torch.Tensor: Selected action.
        """
        action, _, _ = self.get_action_logprob_entropy(state)
        return action

We can initialize our agent as follows.

In [ ]:
agent = ACAgent().to(device)

We can do a sanity check for the agent implementation by executing the code below.

In [ ]:
test_x = torch.zeros(10, envs.single_observation_space.shape[0], device=device)
test_probs = Categorical(F.one_hot(torch.arange(0, envs.single_action_space.n, device=device), num_classes=envs.single_action_space.n))
test_actions = torch.tensor([0, 1, 2, 3, 4, 5], device=device)
test_logprob = torch.tensor([-1.1921e-07, -1.1921e-07, -1.1921e-07, -1.1921e-07, -1.1921e-07, -1.1921e-07], device=device)

assert list(agent.get_value(test_x).shape) == [10, 1]
assert list(agent.get_probs(test_x).logits.shape) == [10, envs.single_action_space.n]
assert (agent.get_action(test_probs) == test_actions).all()
assert torch.allclose(agent.get_action_logprob(test_probs, test_actions), test_logprob)

## Actor-Critic Training with Proximal Policy Optimization (PPO)

Actor-critic training with Proximal Policy Optimization (PPO) is a reinforcement learning framework that enhances both the stability and efficiency of policy learning.

### Training Process

1. **Rollout**: The agent interacts with the environment for a predefined number of steps, known as the rollout. During this phase, the actor selects actions based on its current policy, while the agent gathers states, actions, rewards, and other relevant information, which are stored in a rollout buffer. This collected data forms the basis for subsequent policy and value updates.

2. **Advantage Estimation**: After completing the rollout, the agent computes advantages using the Generalized Advantage Estimation (GAE) method. This technique produces stable and low-variance estimates of the advantage function, indicating how much better or worse an action performed compared to the expected return. GAE improves learning efficiency by providing more accurate feedback for policy updates.

3. **Updating Actor and Critic**:

   - **Policy Update**: The actor's policy is refined using the computed advantages. PPO employs a clipped surrogate objective to limit the magnitude of policy updates, ensuring that changes remain within a safe range. This clipping mechanism fosters stable learning and reduces the risk of drastic performance drops.

   - **Value Update**: The critic's value function is updated to minimize the difference between predicted values and actual returns, typically using mean squared error (MSE) loss. This adjustment allows the critic to provide reliable feedback to the actor, enhancing the overall learning process.

   - **Entropy Regularization**: To promote exploration, PPO includes an entropy term in its objective function. This term discourages certainty in action selection, encouraging the agent to explore a wider range of actions and preventing premature convergence on suboptimal policies.

### Rollout

#### Rollout Buffer

In the rollout buffer, we gather experiences from the environment over a fixed horizon of `ROLLOUT_STEPS`. After the rollout, we use this data to update the agent’s policy and value function; then the buffer is discarded. (This contrasts with DQN-style replay buffers, which reuse transitions across many updates.)

A transition at time $t$ can be summarized as
$$
(S_t, A_t, R_{t+1}, S_{t+1}).
$$

The rollout buffer stores the quantities needed for PPO updates, including:

- **States**: $S_t$
- **Actions**: $A_t$
- **Rewards**: $R_{t+1}$
- **Done Flags**: episode termination indicators
- **Log probabilities**: $\log \pi(A_t|S_t,\boldsymbol{\theta}_{\text{old}})$ under the behavior (old) policy
- **Values**: $\hat v(S_t,\mathbf{w}_{\text{old}})$ from the critic

Because we use `NUM_ENVS` parallel environments, each stored tensor has shape `(ROLLOUT_STEPS, NUM_ENVS)` (plus any observation/action dimensions). This lets us collect diverse experience efficiently and reduces correlation compared to using a single environment.

In [ ]:
states = torch.zeros((ROLLOUT_STEPS, NUM_ENVS) + envs.single_observation_space.shape).to(device)
actions = torch.zeros((ROLLOUT_STEPS, NUM_ENVS) + envs.single_action_space.shape).to(device)
rewards = torch.zeros((ROLLOUT_STEPS, NUM_ENVS)).to(device)
dones = torch.zeros((ROLLOUT_STEPS, NUM_ENVS)).to(device)

logprobs = torch.zeros((ROLLOUT_STEPS, NUM_ENVS)).to(device)
values = torch.zeros((ROLLOUT_STEPS, NUM_ENVS)).to(device)

#### Reward Normalization

To stabilize training, it is common to **adjust and scale** rewards so their magnitudes stay in a consistent range (e.g., by standardizing or by min–max scaling). This can reduce sensitivity to reward scale and improve optimization.

Here, we compute the minimum and maximum rewards of the environment, which we later use for min–max scaling.

In [ ]:
floors = 5
max_waiting = 3
max_in_ele = 10
in_ele_penalty = 0.75
people_waiting_penalty = 3.0
reward_delivered = 30

min_reward = - in_ele_penalty * max_in_ele - people_waiting_penalty * max_waiting * floors
max_reward = max_in_ele * reward_delivered

print("Minimum reward:", min_reward)
print("Maximum reward:", max_reward)

#### Batch, Mini Batch, and Iterations

After completing a rollout of length $T$ across $N$ parallel environments, we obtain one batch of data with a size of $T \times N$. This total represents the batch size. We train the actor-critic agent using this batch by splitting the data into minibatches and training for $E$ epochs.

The total number of iterations or updates is calculated by dividing the total number of interaction steps by the batch size.

**Note:** The batch size mentioned here refers specifically to the rollout batch size. In the code implementation, we use `batch_size` to denote the number of batched inputs/outputs/data, which may differ from the rollout batch size.

In [ ]:
BATCH_SIZE = ROLLOUT_STEPS * NUM_ENVS
MINI_BATCH_SIZE = BATCH_SIZE // NUM_MINI_BATCHES
NUM_ITERATIONS = TOTAL_STEPS // BATCH_SIZE

### Task 2: Computing the Advantages

The (state-)action advantage for a policy $\pi$ is
$$
A_\pi(s,a) \doteq q_\pi(s,a) - v_\pi(s),
$$
where the value functions are defined by
$$
 v_\pi(s) \doteq \mathbb{E}_\pi\left[G_t \mid S_t=s\right],
\qquad
 q_\pi(s,a) \doteq \mathbb{E}_\pi\left[G_t \mid S_t=s, A_t=a\right].
$$

In actor–critic methods we often estimate an advantage at each time step based on an approximate value function $\hat v(s,\mathbf{w})$:
$$
\delta_t \doteq R_{t+1} + \gamma\,\hat v(S_{t+1},\mathbf{w}) - \hat v(S_t,\mathbf{w}).
$$
When an episode terminates at $t+1$, we do not bootstrap, so the next-state value term is masked out.

In PPO we typically use **Generalized Advantage Estimation (GAE)**. One common recursive form is
$$
A^{GAE(\lambda)}_t = \delta_t + \gamma\lambda\,(\text{nonterminal}_{t+1})\,A^{GAE(\lambda)}_{t+1},
$$
which expands to the discounted sum of estimated advantages:
$$
A^{GAE(\lambda)}_t = \sum_{l=0}^{T-t-1} (\gamma\lambda)^l\,\delta_{t+l}.
$$

In this task, you will implement the estimated advantages $\delta_t$. In code, the terminal mask is provided as `next_nonterminal` (equal to $1$ for nonterminal next states and $0$ for terminal next states).

In [ ]:
def get_deltas(rewards, values, next_values, next_nonterminal, gamma):
    """Compute the estimated advantage δ_t.

    Args:
        rewards (torch.Tensor): Rewards R_{t+1}, shape: (batch_size,).
        values (torch.Tensor): Value predictions v(S_t), shape: (batch_size,).
        next_values (torch.Tensor): Value predictions v(S_{t+1}), shape: (batch_size,).
        next_nonterminal (torch.Tensor): 0/1 mask for nonterminal next states, shape: (batch_size,).
        gamma (float): Discount factor γ.

    Returns:
        torch.Tensor: Estimated advantages δ_t, shape: (batch_size,).
    """
    ### -------------- TASK 2 ------------ ###
    # YOUR CODE STARTS HERE
    raise NotImplementedError()
    # YOUR CODE ENDS HERE


In [ ]:
# Test get_deltas
dummy_rewards = torch.ones(3)
dummy_values = torch.tensor([4,5,6])
dummy_next_values = torch.arange(3)
dummy_next_nonterminal = torch.tensor([1,0,1])
dummy_deltas = torch.tensor([-3., -4., -4.8])
assert torch.allclose(get_deltas(dummy_rewards, dummy_values, dummy_next_values, dummy_next_nonterminal, gamma=0.1), dummy_deltas)

### Task 3: Updating the Actor and Critic Networks

After computing the advantages and returns, we shuffle the rollout data and divide the data into mini-batches. We then use the mini-batches to update the actor and critic networks.

#### Task 3.1: Compute the Surrogate Policy Objective

PPO updates the policy by maximizing a **clipped surrogate objective** (Schulman et al., 2017):

$$
L^{\text{PPO}}(\boldsymbol{\theta}) = \mathbb{E}_{\pi_{old}}\left[\min\left(\rho_t(\boldsymbol{\theta})\,\hat{A}_t,\;\text{clip}(\rho_t(\boldsymbol{\theta}), 1-\epsilon, 1+\epsilon)\,\hat{A}_t\right)\right],
$$

where $\hat{A}_t$ is an advantage estimate and the probability ratio is

$$
\rho_t(\boldsymbol{\theta}) \doteq \frac{\pi(A_t|S_t,\boldsymbol{\theta})}{\pi(A_t|S_t,\boldsymbol{\theta}_{\text{old}})}.
$$

Here, we use GAE for the advantage estimate, $\hat{A} = A^{GAE(\lambda)}$.

You will implement the `get_ratio` and `get_policy_objective` functions:

- **Task 3.1.1** (`get_ratio`): compute $\rho_t(\boldsymbol{\theta})$. For numerical stability, work in log space:
  $$
  \rho_t(\boldsymbol{\theta}) = \exp\Big(\log \pi(A_t|S_t,\boldsymbol{\theta}) - \log \pi(A_t|S_t,\boldsymbol{\theta}_{\text{old}})\Big).
  $$

- **Task 3.1.2** (`get_policy_objective`): compute $L^{\text{PPO}}(\boldsymbol{\theta})$ given $\rho_t(\boldsymbol{\theta})$ and $\hat{A}_t$.

In [ ]:
def get_ratio(logprob, logprob_old):
    """Compute the probability ratio ρ_t between the new and old policies.

    Args:
        logprob (torch.Tensor): log π(A_t|S_t,θ), shape: (batch_size,).
        logprob_old (torch.Tensor): log π(A_t|S_t,θ_old), shape: (batch_size,).

    Returns:
        torch.Tensor: ρ_t, shape: (batch_size,).
    """
    ### ------------ TASK 3.1.1 ---------- ###
    # YOUR CODE STARTS HERE
    raise NotImplementedError()
    # YOUR CODE ENDS HERE


In [ ]:
# Test get_ratio
dummy_logprob = torch.tensor([0.1, 0.9])
dummy_logprob_old = torch.tensor([0.5, 0.1])
dummy_ratio = torch.tensor([0.6703, 2.2255])
assert torch.allclose(get_ratio(dummy_logprob, dummy_logprob_old), dummy_ratio, rtol=1e-4)

In [ ]:
def get_policy_objective(advantages, ratio, clip_coeff=CLIP_COEF):
    """Compute the clipped surrogate policy objective.

    Args:
        advantages (torch.Tensor): The advantage estimates, shape: (batch_size,).
        ratio (torch.Tensor): The probability ratio of the new policy to the old policy,
                             shape: (batch_size,).
        clip_coeff (float, optional): The clipping coefficient for the policy objective.
                                       Defaults to CLIP_COEF.

    Returns:
        torch.Tensor: The computed policy objective, a scalar value.
    """
    ### ------------ TASK 3.1.2 ---------- ###
    # YOUR CODE STARTS HERE
    raise NotImplementedError()
    # YOUR CODE ENDS HERE


In [ ]:
# Test get_policy_objective
dummy_advantages = torch.arange(2).float()
assert np.allclose(get_policy_objective(dummy_advantages, dummy_ratio).item(), 0.6)

#### Task 3.2: Compute the Value Loss

The critic is trained to fit a target for the state value. In this notebook, `returns` is used as the target (a bootstrapped estimate of the return). Denote this target by $\hat{G}_t$.

$$
\hat{G}_t = R_{t+1} + \gamma \hat v(S_t,\mathbf{w})
$$

The (unclipped) value loss is the mean squared error:
$$
L^{\text{VF}}_{\text{ori}}(\mathbf{w}) = \hat{\mathbb{E}}\left[\frac{1}{2}\left(\hat v(S_t,\mathbf{w}) - \hat{G}_t\right)^2\right].
$$

PPO commonly applies **value-function clipping** for additional stability. Let $\hat v(S_t,\mathbf{w}_{\text{old}})$ be the value prediction from the previous parameters. The clipped prediction is
$$
\hat v^{\text{clip}}(S_t,\mathbf{w}) = \hat v(S_t,\mathbf{w}_{\text{old}}) + \text{clip}\Big(\hat v(S_t,\mathbf{w}) - \hat v(S_t,\mathbf{w}_{\text{old}}), -\varepsilon, +\varepsilon\Big).
$$
The clipped value loss is
$$
L^{\text{VF}}_{\text{clip}}(\mathbf{w}) = \hat{\mathbb{E}}\left[\frac{1}{2}\left(\hat v^{\text{clip}}(S_t,\mathbf{w}) - \hat{G}_t\right)^2\right].
$$

The final value loss takes the maximum of the two:
$$
L^{\text{VF}}(\mathbf{w}) = \max\left(L^{\text{VF}}_{\text{ori}}(\mathbf{w}),\;L^{\text{VF}}_{\text{clip}}(\mathbf{w})\right).
$$

Implement $L^{\text{VF}}(\mathbf{w})$ below.

In [ ]:
def get_value_loss(values, values_old, returns, clip_coeff=CLIP_COEF):
    """Compute the combined value loss with clipping.

    Args:
        values (torch.Tensor): Predicted values from the critic, shape: (batch_size, 1).
        values_old (torch.Tensor): Old predicted values from the critic, shape: (batch_size, 1).
        returns (torch.Tensor): Computed returns for the corresponding states, shape: (batch_size, 1).
        clip_coeff (float, optional): The clipping coefficient for the value loss.
                                       Defaults to CLIP_COEF.

    Returns:
        torch.Tensor: The combined value loss, a scalar value.
    """
    ### ------------- TASK 3.2 ----------- ###
    # YOUR CODE STARTS HERE
    raise NotImplementedError()
    # YOUR CODE ENDS HERE


In [ ]:
# Test get_value_loss
dummy_values = torch.tensor([1,2,3]).float()
dummy_values_old = torch.tensor([4,5,6]).float()
dummy_returns = torch.tensor([7,8,9]).float()
assert np.allclose(get_value_loss(dummy_values, dummy_values_old, dummy_returns).item(), 18)

#### Compute Entropy Objective

The entropy of a stochastic policy measures how random its action selection is. Higher entropy encourages exploration and can help prevent premature convergence to a suboptimal deterministic policy.

For a categorical policy $\pi(\cdot|S_t,\boldsymbol{\theta})$, the per-state entropy is
$$
H(\pi(\cdot|S_t,\boldsymbol{\theta})) \doteq -\sum_a \pi(a|S_t,\boldsymbol{\theta})\,\log \pi(a|S_t,\boldsymbol{\theta}).
$$
In PPO we typically maximize the average entropy across the batch:
$$
\hat{\mathbb{E}}_\pi\left[H(\pi(\cdot|S_t,\boldsymbol{\theta}))\right].
$$

The implementation of the entropy objective is given below.


In [ ]:
def get_entropy_objective(entropy):
    """Compute the entropy objective.

    Args:
        entropy (torch.Tensor): Entropy values for the action distribution, shape: (batch_size,).

    Returns:
        torch.Tensor: The computed entropy objective, a scalar value.
    """
    return entropy.mean()  # Return the average entropy

#### Task 3.3: Compute the Total Loss

The PPO update combines three goals:

1. **Improve the policy** by maximizing the clipped surrogate objective $L^{\text{PPO}}(\boldsymbol{\theta})$.
2. **Fit the value function** by minimizing the value loss $L^{\text{VF}}(\mathbf{w})$.
3. **Encourage exploration** by maximizing an entropy objective $\hat{\mathbb{E}}_\pi\left[H(\pi(\cdot|S_t,\boldsymbol{\theta}))\right]$.

When using gradient descent, we convert maximization objectives into minimization objectives by negating them. A common total loss is:

$$
L^{\text{PPO}}(\boldsymbol{\theta},\mathbf{w}) = -L^{\text{PPO}}(\boldsymbol{\theta}) + c_1\,L^{\text{VF}}(\mathbf{w}) - c_2\,\hat{\mathbb{E}}_\pi\left[H\big(\pi(\cdot|S_t,\boldsymbol{\theta})\big)\right],
$$

where $c_1$ is `VALUE_LOSS_COEF` and $c_2$ is `ENTROPY_COEF`.

Implement this total loss below.

In [ ]:
def get_total_loss(policy_objective, value_loss, entropy_objective, value_loss_coeff=VALUE_LOSS_COEF, entropy_coeff=ENTROPY_COEF):
    """Compute the total loss for the actor-critic agent.

    Args:
        policy_objective (torch.Tensor): The policy objective, a scalar value.
        value_loss (torch.Tensor): The computed value loss, a scalar value.
        entropy_objective (torch.Tensor): The computed entropy objective, a scalar value.
        value_loss_coeff (float, optional): Coefficient for scaling the value loss. Defaults to VALUE_LOSS_COEF.
        entropy_coeff (float, optional): Coefficient for scaling the entropy loss. Defaults to ENTROPY_COEF.

    Returns:
        torch.Tensor: The total computed loss, a scalar value.
    """
    ### ------------- TASK 3.3 ----------- ###
    # YOUR CODE STARTS HERE
    raise NotImplementedError()
    # YOUR CODE ENDS HERE


In [ ]:
# Test get_total_loss
dummy_policy_objective = torch.tensor(1)
dummy_value_loss = torch.tensor(2)
dummy_entropy_loss = torch.tensor(3)
assert np.allclose(get_total_loss(dummy_policy_objective, dummy_value_loss, dummy_entropy_loss).item(), -0.03)

#### Training

Run the following code to train your agent.

**Note:** As a preliminary check, your agent should achieve an episodic total reward of approximately -4000 by the 300th episodes and around -3000 by the 1500th episodes. If this is not the case, it may indicate issues with your implementation.

In [ ]:
# Enable inline plotting
%matplotlib inline

# Initialize global step counter and reset the environment
global_step = 0
initial_state, _ = envs.reset()
state = torch.Tensor(initial_state).to(device)
done = torch.zeros(NUM_ENVS).to(device)

# Set up progress tracking
progress_bar = tqdm.tqdm(range(1, NUM_ITERATIONS + 1), postfix={'Total Rewards': 0})
actor_loss_history = []
critic_loss_history = []
entropy_objective_history = []

reward_history = []
episode_history = []

# Initialize the optimizer for the agent's parameters
optimizer = optim.Adam(agent.parameters(), lr=LEARNING_RATE, eps=1e-5)

for iteration in progress_bar:
    # Adjust the learning rate using a linear decay
    fraction_completed = 1.0 - (iteration - 1.0) / NUM_ITERATIONS
    current_learning_rate = fraction_completed * LEARNING_RATE
    optimizer.param_groups[0]["lr"] = current_learning_rate

    # Perform rollout to gather experience
    for step in range(0, ROLLOUT_STEPS):
        global_step += NUM_ENVS
        states[step] = state
        dones[step] = done

        with torch.no_grad():
            # Get action, log probability, and entropy from the agent
            action, log_probability, _ = agent.get_action_logprob_entropy(state)
            value = agent.get_value(state)
            values[step] = value.flatten()

        actions[step] = action
        logprobs[step] = log_probability

        # Execute action in the environment
        # next_state, reward, done, _, info = envs.step(action.cpu().numpy())
        next_state, reward, terminated, truncated, info = envs.step(action.cpu().numpy())
        done = np.logical_or(terminated, truncated)

        normalized_reward = (reward - min_reward) / (max_reward - min_reward)  # Normalize the reward
        rewards[step] = torch.tensor(normalized_reward).to(device).view(-1)
        state = torch.Tensor(next_state).to(device)
        done = torch.Tensor(done).to(device)


        if "final_info" in info:
            for i, ep_info in enumerate(info["final_info"]):
                if ep_info is not None and "episode" in ep_info:
                    episodic_reward = ep_info["episode"]["r"]
                    reward_history.append(episodic_reward)
                    episode_history.append(global_step)
                    progress_bar.set_postfix({'Total Rewards': episodic_reward})
        elif isinstance(info, dict) and "episode" in info:
            # finished envs mask
            mask = info.get("_episode", None)
            if mask is None:
                mask = info["episode"].get("_r", None)

            if mask is not None:
                # iterate over the finished envs and log their returns
                rs = np.asarray(info["episode"]["r"])
                for episodic_reward in rs[mask]:
                    episodic_reward = float(episodic_reward)
                    reward_history.append(episodic_reward)
                    episode_history.append(global_step)
                    progress_bar.set_postfix({'Total Rewards': episodic_reward})

    # Calculate advantages and returns
    with torch.no_grad():
        next_value = agent.get_value(state).reshape(1, -1)
        advantages = torch.zeros_like(rewards).to(device)

        last_gae_lambda = 0
        for t in reversed(range(ROLLOUT_STEPS)):
            if t == ROLLOUT_STEPS - 1:
                next_non_terminal = 1.0 - done
                next_value = next_value
            else:
                next_non_terminal = 1.0 - dones[t + 1]
                next_value = values[t + 1]

            # Compute delta using the utility function
            delta = get_deltas(rewards[t], values[t], next_value, next_non_terminal, gamma=GAMMA)

            advantages[t] = last_gae_lambda = delta + GAMMA * GAE_LAMBDA * next_non_terminal * last_gae_lambda
        returns = advantages + values

    # Flatten the batch data for processing
    batch_states = states.reshape((-1,) + envs.single_observation_space.shape)
    batch_logprobs = logprobs.reshape(-1)
    batch_actions = actions.reshape((-1,) + envs.single_action_space.shape)
    batch_advantages = advantages.reshape(-1)
    batch_returns = returns.reshape(-1)
    batch_values = values.reshape(-1)

    # Shuffle the batch data to break correlation between samples
    batch_indices = np.arange(BATCH_SIZE)
    total_actor_loss = 0
    total_critic_loss = 0
    total_entropy_objective = 0

    for epoch in range(NUM_EPOCHS):
        np.random.shuffle(batch_indices)
        for start in range(0, BATCH_SIZE, MINI_BATCH_SIZE):
            # Get the indices for the mini-batch
            end = start + MINI_BATCH_SIZE
            mini_batch_indices = batch_indices[start:end]

            mini_batch_advantages = batch_advantages[mini_batch_indices]
            # Normalize advantages to stabilize training
            mini_batch_advantages = (mini_batch_advantages - mini_batch_advantages.mean()) / (mini_batch_advantages.std() + 1e-8)

            # Compute new probabilities and values for the mini-batch
            new_probabilities = agent.get_probs(batch_states[mini_batch_indices])
            new_log_probability = agent.get_action_logprob(new_probabilities, batch_actions.long()[mini_batch_indices])
            entropy = agent.get_entropy(new_probabilities)
            new_value = agent.get_value(batch_states[mini_batch_indices])

            # Calculate the policy loss
            ratio = get_ratio(new_log_probability, batch_logprobs[mini_batch_indices])
            policy_objective = get_policy_objective(mini_batch_advantages, ratio, clip_coeff=CLIP_COEF)
            policy_loss = -policy_objective

            # Calculate the value loss
            value_loss = get_value_loss(new_value.view(-1), batch_values[mini_batch_indices], batch_returns[mini_batch_indices])

            # Calculate the entropy loss
            entropy_objective = get_entropy_objective(entropy)

            # Combine losses to get the total loss
            total_loss = get_total_loss(policy_objective, value_loss, entropy_objective, value_loss_coeff=VALUE_LOSS_COEF, entropy_coeff=ENTROPY_COEF)

            optimizer.zero_grad()
            total_loss.backward()
            # Clip the gradient to stabilize training
            nn.utils.clip_grad_norm_(agent.parameters(), 0.5)
            optimizer.step()

            total_actor_loss += policy_loss.item()
            total_critic_loss += value_loss.item()
            total_entropy_objective += entropy_objective.item()

    actor_loss_history.append(total_actor_loss // NUM_EPOCHS)
    critic_loss_history.append(total_critic_loss // NUM_EPOCHS)
    entropy_objective_history.append(total_entropy_objective // NUM_EPOCHS)

    # Prepare data for live plotting
    data_to_plot = {
        'Total Reward': reward_history,
        'Actor Loss': actor_loss_history,
        'Critic Loss': critic_loss_history,
        'Entropy': entropy_objective_history
    }
    live_plot(data_to_plot)

live_plot(data_to_plot, save_pdf=True, output_file='training_curves.pdf')
# Close the environment after training
envs.close()

Save the agent model by executing the code below.

In [ ]:
torch.save(agent.state_dict(), "model.pth")

## Agent Evaluation

After training the agent, you can evaluate your agent locally using the following code.

In [ ]:
env = create_elevator_env()

agent_test = ACAgent().to(device)

agent_test.load_state_dict(torch.load("model.pth", map_location=device))

num_episodes_to_run = 10
rewards = []

for episode in tqdm.tqdm(range(num_episodes_to_run)):
    total_reward = 0

    state, _ = env.reset()
    state = torch.tensor(state, dtype=torch.float32, device=device)
    while True:
        state_tensor = state

        with torch.no_grad():
            action = agent_test.act(state_tensor)

        next_state, reward, terminated, truncated, info = env.step(action.cpu().item())
        done = np.logical_or(terminated, truncated)

        total_reward += reward

        state = torch.tensor(next_state, dtype=torch.float32, device=device)

        if done:
            break

    rewards.append(total_reward)

env.close()

print(f"\nMean Rewards: {np.mean(rewards)}\n")

#### Task 4: Evaluating your Agent in Coursemology

Run the code below to generate a `get_model` function that stores your trained agent. Then, copy and paste both your `ACAgent` class and the generated function into Coursemology to evaluate your trained agent.

In [ ]:
from utils import generate_torch_loader_snippet

agent_test = ACAgent().to(device)
agent_test.load_state_dict(torch.load("model.pth", map_location=device))

# Generate and print the code snippet for loading the model using state_dict
print(generate_torch_loader_snippet(agent_test, prefer="state_dict"))

# If the above does not work, you can use the following method:
# Generate and print the code snippet for loading the model using pickle
# print(generate_torch_loader_snippet(agent_test, prefer="pickle"))